# Aggregation over simulations

In [ ]:
import subprocess
import os

filename = 'aggregate.txt'

# Check if the file exists first to avoid confusing errors
if not os.path.exists(filename):
    print(f"Error: Could not find '{filename}'. Make sure it's in the same directory.")
    exit(1)

with open(filename, 'r') as file:
    for line in file:
        # Strip whitespace and split the line into parts
        parts = line.strip().split()
        
        # Skip empty lines or malformed lines
        if len(parts) != 2:
            continue
            
        scenario, model = parts
        
        print(f"Aggregate Scenario: {scenario} with Model: {model}")
        
        # Build the command exactly as it was in Bash
        cmd = [
            "python", "lou_simulation_hpc_mp_old.py",
            "--scenario", scenario,
            # NOTE: If you changed this to --models in your argparse earlier, 
            # make sure to update this string to "--models" 
            "--model", model, 
            "--aggregate_only"
        ]
        
        # Execute the command
        try:
            # check=True ensures that if a script fails, it raises an exception
            subprocess.run(cmd, check=True)
        except subprocess.CalledProcessError as e:
            print(f"  [X] Error running aggregation for {scenario} - {model}. Moving to next...")

print("\nAll individual aggregations complete!")

# Cross-scenario aggregation

In [ ]:
import subprocess
from collections import defaultdict

# Dictionary to hold lists of models for each scenario
scenario_models = defaultdict(list)

# 1. Parse the text file
with open('aggregate.txt', 'r') as file:
    for line in file:
        parts = line.strip().split()
        if len(parts) == 2:
            scenario, model = parts
            scenario_models[scenario].append(model)

# 2. Run the command for each grouped scenario
for scenario, models in scenario_models.items():
    print(f"\n{'='*40}")
    print(f"Cross-Aggregating Scenario: {scenario}")
    print(f"Models: {', '.join(models)}")
    print(f"{'='*40}")
    
    # Build the terminal command
    cmd = [
        "python", "lou_simulation_hpc_mp_old.py",
        "--scenario", scenario,
        "--models"
    ] + models + ["--cross_aggregate"]
    
    # Execute the command
    subprocess.run(cmd)